# 35 · Capstone: ecommerce return review

Combine tools, local FAISS retrieval, deterministic eligibility and structured output. The agent proposes a review decision; it never moves money.

**Runtime:** ADK 2.10.0 on `aadhi`, existing `dataengenv` kernel and ADC.

**Cost:** About 3 Lite calls; local FAISS. Synthetic ecommerce data only. Run cells from top to bottom; each notebook creates its own sessions.


In [ ]:
from pathlib import Path
import sys
# Works when Jupyter starts in GoogleADK or its parent folder.
root = Path.cwd() if (Path.cwd() / "course.py").exists() else Path.cwd() / "GoogleADK"
if not (root / "course.py").exists():
    raise RuntimeError("Open this notebook with GoogleADK as the working directory.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from course import (ROOT, CONFIG, MODEL, PROJECT, LOCATION, SKIP_SSL, DATA,
                    llm, gemini, make_lab, ask, final_text, token_usage,
                    get_order, get_product, get_customer, types)
print(f"Project={PROJECT} | location={LOCATION} | model={MODEL} | TLS bypass={SKIP_SSL}")


In [ ]:
from pydantic import BaseModel
from google.adk.agents import SequentialAgent
from local_rag import PolicyIndex
policies = PolicyIndex(DATA["policies"])

def return_evidence(order_id: str) -> dict:
    """Get order, membership, local policy evidence and deterministic return eligibility."""
    order = get_order(order_id)
    if "error" in order:
        return order
    customer = get_customer(order["customer_id"])
    window = 14 if customer.get("tier") == "gold" else 7
    age = order.get("days_since_delivery")
    eligible = order["status"] == "delivered" and age is not None and 0 <= age <= window
    return {"order": order, "customer": customer, "window_days": window,
            "within_return_window": eligible,
            "policies": policies.search("returns refunds invoice unused gold", k=1),
            "remaining_checks": ["unused condition", "original invoice", "human approval"]}

class ReturnReview(BaseModel):
    order_id: str
    within_return_window: bool
    window_days: int
    proposed_amount: float
    currency: str
    remaining_checks: list[str]
    source_ids: list[str]

collector = llm("evidence_collector", "Use return_evidence for the requested order. Return all evidence faithfully.",
                tools=[return_evidence], output_key="evidence")
reviewer = llm("return_reviewer", "Produce a return review from {evidence}. Preserve deterministic eligibility. "
               "The order total is only a proposed amount; no refund is approved. Include remaining checks and source IDs.",
               output_schema=ReturnReview)
flow = SequentialAgent(name="return_review", sub_agents=[collector, reviewer])
lab = await make_lab(flow)
events = await ask(lab, "Review a return request for O1001.")
review = ReturnReview.model_validate_json(final_text(events))
assert review.order_id == "O1001" and review.within_return_window and review.window_days == 14
print(review.model_dump())


## Try it

Add artifact export and the confirmation tool from notebook 20. Keep policy retrieval, business calculations and human approval as separate steps.


## Reference

[Official ADK documentation](https://adk.dev/). Shared configuration: `config.json`. No environment activation or login cells are needed.
